# Dynamics 04 — Transparence des programmes et issue du Dilemme du prisonnier one-shot

**Navigation** : [<< Dynamics-03-Sympathie-vs-Engagement-CSharp](Dynamics-03-Sympathie-vs-Engagement-CSharp.ipynb) | [Version Python](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) | [Index](README.md) | [compagnon formel 04b](Dynamics-04b-FairBot-Loeb-Lean.ipynb) | [Dynamics-05-Agents-Bornes-CSharp >>](Dynamics-05-Agents-Bornes-CSharp.ipynb)

Quand l'agent adverse peut **lire** votre strategie, l'equilibre de Nash `(D,D)` du PD classique
cesse d'etre l'unique verdict : la transparence du programme change l'issue. Ce notebook
explore cette frontiere entre jeu sous forme normale et jeu sous forme extensive ou les
joueurs sont des **programmes** (heuristiques `_toy`, cf. header §0) bornes et totaux.

References : Shoham & Leyton-Brown (2009) §3.4.2 ; Osborne (2004) §3.2 ; Axelrod & Hamilton (*Science*, 1981).

## 0. Sources primaires et conventions de nommage

**Sources publiées citées dans ce notebook** :

- **Shoham & Leyton-Brown**, *Multiagent Systems: Algorithmic, Game-Theoretic, and Logical Foundations* (Cambridge University Press, 2009), chapitre **§3.4.2 « Iterated Dominance and the Prisoner's Dilemma »**. C'est la référence pour la paramétrisation canonique du Dilemme du prisonnier `(T=5, R=3, P=1, S=0)` et la condition `2R > T + S` qui distingue un PD **strict** d'un jeu à alternance (cf. exercice 3 de Shoham-Leyton-Brown §3.4).
- **Osborne**, *An Introduction to Game Theory* (Oxford University Press, 2004), chapitre **§3.2 « Strictly Dominant Strategies »**. La dominance stricte de `D` sur `C` pour les deux joueurs — peu importe l'action de l'adversaire — établit `(D, D)` comme équilibre de Nash **unique** en stratégies pures.
- **Axelrod & Hamilton**, « The Evolution of Cooperation » (*Science*, vol. 211, n° 4489, 1981, pp. 1390-1396). Cité pour `TitForTat` (Axelrod, 1980) dans le contexte des jeux répétés. `TitForTat` n'est **pas** implémenté dans le moteur du présent notebook — il figure en exercice 1 pour situer la transparence des programmes par rapport à la littérature des stratégies itérées.
- **Critch, Dennis & Russell**, « Cooperative and uncooperative institution designs: Surprises and problems in open-source game theory » (2022, [arXiv:2208.07006](https://arxiv.org/abs/2208.07006)), **§3 « Inspection games »** et **Open Problem 3** (p. 18). C'est la référence canonique pour les définitions de `DUPOC(k)` (*Do-Unto-Other-Player with Probability k of Cooperation*) et `CUPOD(k)` (*Cooperate-Unless-Player-is-Defecting*) dans le cadre de l'inspection mutuelle de programmes. **Open Problem 3** pose la conjecture sur le comportement asymptotique de `DUPOC(k)` vs `CUPOD(k)` au-delà de quelques itérations ; ce notebook l'illustre mais ne la résout pas (cf. la section 8, « Limites et Open Problems »). Les résultats numérotés du papier sont cités avec leur page : **Propositions 3.1 et 3.2** (p. 11) établissent que `CUPOD(k)` n'exploite jamais son adversaire et que `DUPOD(k)` n'est jamais exploité ; le **Lemma 3.6** (*PBLT — Parametric Bounded Löb Theorem*, p. 13) est l'ingrédient de robustesse des agents raisonnant sur des preuves de longueur bornée ; le **Theorem 3.4** (p. 13) montre `outcome(CUPOD(k), CUPOD(k)) == (D, D)` — deux bots prudents mutuellement transparents peuvent néanmoins se défaire, un contre-intuitif central de l'open-source game theory.


- **Cooper, Oesterheld & Conitzer**, « Characterising Simulation-Based Program Equilibria » (2024, [arXiv:2412.14570](https://arxiv.org/abs/2412.14570), v2 2025). La référence pour les programmes fondés sur **simulation** (plutôt que sur preuve) — les `εGroundedπBot`. Trois résultats numérotés cadrent la section 8 : le **Theorem 1** (p. 3) restitue le folk theorem de **Tennenholtz (2004)** pour les jeux-programmes — tout profil de paiements faisable et individuellement rationnel est atteignable en équilibre ; les **Theorem 5** (p. 5) et **Corollary 1** (p. 6) établissent un folk theorem complet pour les programmes *corrélés* (aléa partagé) ; les **Propositions 1-2** (p. 7) et la discussion de la p. 9 montrent que ce folk theorem **échoue sans aléa partagé**, sauf à utilités additivement séparables (**Theorem 7**, p. 8).
- **Barasz, Christiano, Fallenstein, Herreshoff, LaVictore & Yudkowsky**, « Robust Cooperation in the Prisoner's Dilemma: Program Equilibrium via Provability Logic » (2014, [arXiv:1401.5577](https://arxiv.org/abs/1401.5577)). Le socle logique des agents sur preuves : **Theorem 3.1** (p. 8) `PA ⊢ [FairBot(FairBot) = C]` — deux FairBot mutuellement lisibles coopèrent par Löb — et **Theorem 3.2** (p. 9) : PrudentBot est inexploitable, coopère avec lui-même et avec FairBot. Les `FairBot_toy` et `PrudentBot_toy` de ce notebook simulent ce comportement par inspection textuelle bornée, sans la logique de prouvabilité (cf. section 8).
- **Aumann**, « Subjectivity and Correlation in Probabilistic * Strategic Settings » (1974, [pdf IJGT 1(2): 67-96](https://link.springer.com/article/10.1007/BF01755010)) et « Correlated Equilibrium as an Expression of Bayesian Rationality » (1987, *Econometrica* 55(1): 1-18). Fonde le concept d'**equilibre correle** : un mediator public communique aux joueurs des recommandations d'action tirees d'une distribution jointe ; si chaque joueur suit la recommandation *qui lui est adressee*, l'equilibre peut etre strictement meilleur que tout equilibre de Nash. La **transparence des programmes** du present notebook peut etre lue comme un mediator implicite (cf. cellule d'interpretation A ci-apres) : la lecture du source de l'adversaire *est* la recommandation, et l'inspection mutuelle joue le role du signal public.
- **Nash**, « Equilibrium Points in *N*-Person Games » (1950, *PNAS* 36(1): 48-49, [pdf](https://www.pnas.org/doi/pdf/10.1073/pnas.36.1.48)). Definit formellement l'**equilibre de Nash** en strategies mixtes pour N joueurs : le point ou aucun joueur n'augmente son gain esperer en unilateralement changer de strategie. La section 1 (« Le Dilemme du prisonnier classique ») utilise cette definition pour montrer que `(D, D)` reste l'unique equilibre de Nash **en strategies pures** du PD classique `(T=5, R=3, P=1, S=0)`. La transparence des programmes (à partir de la section 1, « Lecture : equilibre correle et transparence comme mediator ») sort de ce cadre : les bots ne choisissent plus une strategie mixtee sur l'ensemble des actions pures, ils *lisent le programme adverse* et conditionnent leur action au source.

**Conventions de nommage** :

- Les bots implémentés dans ce notebook sont des **heuristiques ad hoc** (locales au notebook), suffixés `_toy` : `CooperateBot_toy`, `DefectBot_toy`, `FairBot_toy`, `CUPOD_toy`, `PrudentBot_toy`. Ce suffixe marque la frontière entre la pédagogie locale et la littérature publiée. Aucune des stratégies `_toy` n'est une reproduction d'une stratégie publiée (la ressemblance FairBot/DUPOC et CUPOD_toy/CUPOD est **intentionnelle**, pas une citation : `DUPOC` et `CUPOD` sont des concepts théoriques définis par Critch-Dennis-Russell 2022 §3, et aucune instance de ces stratégies n'est reproduite ici).
- `TitForTat` (Axelrod 1980) **n'a pas de suffixe `_toy`** parce qu'il figure comme référence externe, jamais instancié dans le moteur. Il est mentionné en exercice 1.

Cette séparation est sémantique : `_toy` signale au lecteur que le bot est une heuristique pédagogique, pas une reproduction de stratégie publiée.


## Objectifs d'apprentissage

- Vérifier que `(D,D)` est l'unique équilibre de Nash du Dilemme du prisonnier classique (Osborne §3.2).
- Définir un `ProgramAgent` borné dont l'action dépend de la représentation de l'adversaire.
- Implémenter cinq bots `_toy` : `CooperateBot_toy`, `DefectBot_toy`, `FairBot_toy`/`DUPOC`, `CUPOD_toy` et `PrudentBot_toy` (cf. header §0 : `_toy` désigne une heuristique locale, pas une reproduction de stratégie publiée).
- Produire la matrice de confrontation et interpréter trois phénomènes : coopération mutuelle, exploitation et défection paradoxale.
- Distinguer les états d'arrêt `play`, `depth` et `timeout` selon que la confrontation aboutit, atteint la profondeur maximale ou épuise son budget avant l'appel.
- Contrôler la matrice avec un second calcul indépendant de `simulate_payoff`.
- Comparer répétition/Axelrod, preuve formelle (Lean), simulation et transparence.


### Prérequis

- `GameTheory-06` (Evolution of Trust — Axelrod).
- `GameTheory-06c` (Folk theorem des jeux répétés).
- `GameTheory-06d` (statique comparative sympathie vs engagement).


## 1. Le Dilemme du prisonnier classique : `(D,D)` est l'unique équilibre de Nash


### Lecture : equilibre correle et transparence comme mediator

L'equilibre de Nash (cf. la section 1, « Le Dilemme du prisonnier classique ») suppose que chaque joueur choisit sa strategie en
fonction de l'histoire du jeu, **sans communication** avec l'adversaire. L'equilibre
correle d'**Aumann 1974** etend ce cadre : un *mediator* public tire un profil d'actions
`s = (s_1, ..., s_n)` selon une distribution jointe connue, et communique a chaque
joueur *sa propre* recommandation `s_i` (jamais celle des autres). Un equilibre
correle est une distribution jointe telle que, pour chaque joueur, suivre la
recommandation est optimal *conditionnellement* a la recommandation recue.

La portee pratique : avec un mediator, on peut realiser des issues **strictement
meilleures** que le meilleur equilibre de Nash. L'exemple canonique est le jeu de
Bach ou Stravinsky (coordination pure) ; on en retient ici la version **symetrique**, ou
les deux joueurs ont les memes preferences :

| | Bach | Stravinsky |
|---|---|---|
| **Bach** | 1, 1 | 0, 0 |
| **Stravinsky** | 0, 0 | 1, 1 |

Ce jeu admet deux **equilibres de Nash purs** — `(Bach, Bach)` et `(Stravinsky, Stravinsky)` —
de gain 1 chacun — et un **equilibre mixte symetrique** dans lequel chaque joueur tire Bach ou
Stravinsky avec probabilite 1/2, de gain attendu 1/2. Un mediator qui recommande toujours
`(Bach, Bach)` realise l'un des equilibres purs : son gain (1) depasse le gain attendu de
l'equilibre mixte (1/2).

Le Bach ou Stravinsky canonique n'est pas symetrique : chaque joueur prefere une
coordination differente, ce qui donne des gains asymetriques — `(2, 1)` et `(1, 2)` sur les
deux equilibres purs, et un gain attendu `2/3` a chacun a l'equilibre mixte. La version
symetrique ci-dessus est celle ou le gain de l'equilibre mixte se calcule immediatement et ou
l'amelioration par un mediator porte sur les **deux** joueurs ; la these generale — un
mediator peut realiser des issues que l'equilibre de Nash ne permet pas — reste inchangee.

**Lecture du notebook** : la transparence des programmes peut etre lue comme un
*mediator implicite*. La source de l'adversaire est observable publiquement ; la lire
equivaut a recevoir une recommandation qui depend de la *strategie* (le programme)
plutot que d'un signal exogene. L'inspection mutuelle joue le role du signal public.
Mais la distribution jointe n'est pas fixee par le jeu : elle emerge de la combinatoire
des programmes. Le present notebook illustre ce point : la matrice 5x5 fait emerger
des issues `(C, C)` qui ne sont **pas** des equlibres de Nash du PD classique
(le seul equilibre de Nash en strategies pures est `(D, D)`), et qui **ne sont pas
non plus** des equilibres correles au sens d'Aumann -- la distribution jointe
depend de la lecture des sources par chaque bot, pas d'un mediator explicite.

**Sources** : Aumann 1974 §2 (definition de l'equilibre correle), Aumann 1987 §3
(rationalite bayesienne comme fondement), Nash 1950 (definition de l'equilibre de
Nash comme cas particulier ou le mediator est absent). Voir la section 0 (« Sources primaires et conventions de nommage ») pour les
references completes.


On pose la matrice canonique `(T=5, R=3, P=1, S=0)` avec `T > R > P > S` et `2R > T + S`.

| | C | D |
|---|---|---|
| **C** | R, R | S, T |
| **D** | T, S | P, P |

Best response : **D** domine **C** pour chaque joueur, peu importe l'action adverse.
L'unique équilibre de Nash est `(D,D)`.


In [1]:
// Illustration 1 — paiements canoniques du Dilemme du prisonnier (T=5, R=3, P=1, S=0)
const int T = 5, R = 3, P = 1, S = 0;

if (!(T > R && R > P && P > S))
    throw new InvalidOperationException("Parametres PD non canoniques");
if (!(2 * R > T + S))
    throw new InvalidOperationException("Cooperation mutuelle > alternance (stricte PD)");

int PayoffSelf(string myAction, string otherAction) => myAction == "C"
    ? (otherAction == "C" ? R : S)
    : (otherAction == "C" ? T : P);

foreach (string me in new[] { "C", "D" })
{
    foreach (string other in new[] { "C", "D" })
    {
        Console.WriteLine($"self={me,-5} other={other,-5} -> {PayoffSelf(me, other)}");
    }
}

foreach (string myAction in new[] { "C", "D" })
{
    foreach (string other in new[] { "C", "D" })
    {
        // Dominance STRICTE : le gain de D doit surpasser (et non seulement égaler) celui de C.
        if (!(PayoffSelf("D", other) > PayoffSelf("C", other)))
            throw new InvalidOperationException($"D ne domine PAS strictement C : me={myAction} other={other} (D={PayoffSelf("D", other)} vs C={PayoffSelf("C", other)})");
    }
}
Console.WriteLine("D domine C (PD stricte). Equilibre de Nash unique : (D,D).");


self=C     other=C     -> 3


self=C     other=D     -> 0


self=D     other=C     -> 5


self=D     other=D     -> 1


D domine C (PD stricte). Equilibre de Nash unique : (D,D).


## 2. `ProgramAgent` : un agent qui raisonne sur la **représentation** de l'adversaire


## 2. `ProgramAgent` : un agent qui raisonne sur la **représentation** de l'adversaire


On transforme le jeu sous forme normale en jeu sous forme extensive : chaque joueur est un
programme qui reçoit en argument la **représentation textuelle** (le code) de l'adversaire
et choisit son action. Cette transparence est la **clé** qui change l'issue.

Le moteur est **one-shot** : les deux délégués d'une confrontation sont appelés **une seule
fois**. La descente récursive du moteur transporte alors la profondeur et le budget de niveau
en niveau, mais elle **s'arrête avant** la branche de repli : la condition
`depth + 1 < MaxDepth` plafonne la descente à la profondeur `MaxDepth - 1`.

Deux bornes distinctes sont fixées **avant l'appel au bot** (garde d'entrée). La borne
`StepBudget` (1000 par défaut) limite le **nombre de niveaux d'appels autorisés**, pas la
durée d'un appel qui ne termine pas : un délégué qui boucle infiniment bloque le noyau
.NET Interactive dès le premier appel — un notebook n'isole pas chaque cellule.
`SimulationTimeoutException` (cf. la section 6, « Trois états : preuve, absence dans la borne, non-termination ») ne couvre que les budgets épuisés
à l'entrée. Au-delà de `MaxDepth` (3 par défaut), l'action par défaut `(D, D)` et le payoff `(1, 1)` sont
renvoyés avec le statut `depth` ; ce résultat ne constitue pas une preuve d'équilibre.

**Ce que la descente récursive atteint effectivement.** Depuis une profondeur initiale
`depth = 0` et un budget `StepBudget = 1000`, les niveaux visités sont `0`, `1`, `2`
(`MaxDepth - 1 = 2`) et la branche `depth` n'est **pas** atteinte. De même, le budget décroît
d'une unité par niveau (1000, 999, 998) et n'est jamais épuisé par la descente : l'exception de
budget n'est donc levée que lorsqu'un **budget déjà épuisé** est passé à l'entrée. Les deux
états `depth` et `timeout` s'obtiennent ainsi en **injectant** la valeur limite à l'appel
(illustration 7, section 6) : ce sont des cas limites observés, pas des issues que la
descension produit spontanément.

Deux compteurs distincts cohabitent dans le moteur C# : `MaxDepth` (profondeur maximale
CONFIGURABLE, initialisée à 3) et `StepBudget` (budget de pas, initialisé à 1000). La
profondeur borne le **nombre de niveaux de descente** du moteur, le budget borne le
**nombre d'appels autorisés** ; aucun des deux ne borne la **lecture textuelle** faite par un délégué :
le délégué reçoit le texte source **complet** de l'adversaire, sans troncature. Aucune
propriété de sécurité n'est attachée à cette lecture — le carnet ne prétend que décrire ce
que le code implémente.


In [2]:
// Illustration 2 — moteur borne : profondeur maximale configurable et budget de pas distinct
using System;
using System.Collections.Generic;
using System.Linq;

const int T = 5, R = 3, P = 1, S = 0;

int PayoffSelf(string myAction, string otherAction) => myAction == "C"
    ? (otherAction == "C" ? R : S)
    : (otherAction == "C" ? T : P);

sealed class SimulationTimeoutException : Exception
{
    public SimulationTimeoutException(string message) : base(message) { }
}

sealed class ProgramGameEngine
{
    private readonly Func<string, string, int> payoff;

    public ProgramGameEngine(Func<string, string, int> payoffFunction) => payoff = payoffFunction;

    // Profondeur maximale CONFIGURABLE : propriete mutable, initialisee a 3.
    public int MaxDepth { get; set; } = 3;

    // Budget de pas : compteur DISTINCT de la profondeur (valeur par defaut 1000).
    public int StepBudget { get; set; } = 1000;

    // Observateurs du moteur, recalcules a chaque simulation.
    public int BotCallCount { get; private set; }
    public int DescentCount { get; private set; }
    public int DepthBoundHitCount { get; private set; }
    public List<int> StepCapsObserved { get; } = new List<int>();
    public List<int> DepthsObserved { get; } = new List<int>();

    public (string ActionA, string ActionB, int PayoffA, int PayoffB, string Status) SimulatePayoff(
        Func<string, string> playerA, Func<string, string> playerB, string sourceA, string sourceB)
        => SimulatePayoff(playerA, playerB, sourceA, sourceB, StepBudget, 0);

    public (string ActionA, string ActionB, int PayoffA, int PayoffB, string Status) SimulatePayoff(
        Func<string, string> playerA, Func<string, string> playerB, string sourceA, string sourceB, int stepCap, int depth)
        => SimulatePayoffRecursive(playerA, playerB, sourceA, sourceB, stepCap, depth, null, null);

    // Moteur recursif : le budget DECREMENTE et la profondeur sont transmis a chaque appel.
    // Les DELEGUES sont appeles une seule fois par confrontation.
    public (string ActionA, string ActionB, int PayoffA, int PayoffB, string Status) SimulatePayoffRecursive(
        Func<string, string> playerA, Func<string, string> playerB, string sourceA, string sourceB,
        int stepCap, int depth, string actionA, string actionB)
    {
        StepCapsObserved.Add(stepCap);
        DepthsObserved.Add(depth);

        int remaining = stepCap - 1;   // budget DECREMENTE avant tout usage
        if (remaining < 0)
            throw new SimulationTimeoutException($"step_cap epuise avant verdict (step_cap_initial={StepBudget})");

        if (depth >= MaxDepth)
        {
            DepthBoundHitCount++;
            return ("D", "D", payoff("D", "D"), payoff("D", "D"), "depth");
        }

        string actA, actB;
        if (actionA == null || actionB == null)
        {
            BotCallCount += 2;
            actA = playerA(sourceB);
            actB = playerB(sourceA);
        }
        else
        {
            actA = actionA;
            actB = actionB;
        }

        if (depth + 1 < MaxDepth && remaining >= 1)
        {
            DescentCount++;
            var deeper = SimulatePayoffRecursive(playerA, playerB, sourceA, sourceB, remaining, depth + 1, actA, actB);
            if (deeper.Status == "play")
                return deeper;
        }

        return (actA, actB, payoff(actA, actB), payoff(actB, actA), "play");
    }
}

var engine = new ProgramGameEngine(PayoffSelf);
Console.WriteLine($"Moteur pret : MaxDepth={engine.MaxDepth}, StepBudget={engine.StepBudget}, T={T}");
Console.WriteLine("Semantique one-shot : chaque confrontation appelle les deux delegates une seule fois.");

Moteur pret : MaxDepth=3, StepBudget=1000, T=5


Semantique one-shot : chaque confrontation appelle les deux delegates une seule fois.


## 3. Cinq bots-programmes


On definit cinq strategies _toy_ (heuristiques locales au notebook). Chacune est un **délégué**
`Func<string, string>` qui reçoit le **texte source complet** de l'adversaire et renvoie
`"C"` ou `"D"` :

- `CooperateBot_toy` : renvoie `"C"` sans inspecter le texte reçu.
- `DefectBot_toy` : renvoie `"D"` sans inspecter le texte reçu.
- `FairBot_toy` : inspecte le texte reçu ; renvoie `"C"` si ce texte contient la clause
  `return "C"`, et `"D"` sinon.
- `CUPOD_toy` : applique dans ce carnet le **même test textuel** que `FairBot_toy` — présence
  de la clause `return "C"` dans le texte reçu ; il ne tient compte d'aucun tour antérieur, le
  moteur n'étant pas itératif.
- `PrudentBot_toy` : renvoie `"D"` si le texte reçu mentionne `DefectBot_toy` sans mentionner
  `CUPOD_toy` ni `FairBot_toy` ; sinon il renvoie `"C"` si ce texte contient la clause
  `return "C"`, et `"D"` dans les autres cas.

Le suffixe `_toy` signale que ces bots sont des **heuristiques pédagogiques**, pas des
reproductions de stratégies publiées (cf. header §0 pour les conventions de nommage et les
sources primaires).

Ces descriptions correspondent exactement au comportement codé dans la cellule suivante : la
seule information dont dispose un bot est la **chaîne** qu'il reçoit. Aucun bot n'observe
l'action réellement jouée par l'adversaire, ni un historique des tours — ces notions
n'existent pas dans ce moteur.

In [3]:
// Illustration 3 — cinq bots-programmes : delegates et representations textuelles declarees
const string CooperatesMarker = "return \"C\"";

// Sequence d'echappement telle qu'elle apparait dans un source C# valide : backslash + guillemet
string escapedQuote = new string(new[] { (char)92, (char)34 });
string markerEscaped = "return " + escapedQuote + "C" + escapedQuote;   // return \"C\"

Func<string, string> CooperateBot_toy = _otherSource => "C";
Func<string, string> DefectBot_toy = _otherSource => "D";
Func<string, string> FairBot_toy = otherSource => otherSource.Contains(CooperatesMarker) ? "C" : "D";
Func<string, string> CUPOD_toy = otherSource => otherSource.Contains(CooperatesMarker) ? "C" : "D";
Func<string, string> PrudentBot_toy = otherSource =>
    otherSource.Contains("DefectBot_toy") && !otherSource.Contains("CUPOD_toy") && !otherSource.Contains("FairBot_toy")
        ? "D"
        : (otherSource.Contains(CooperatesMarker) ? "C" : "D");

var BOTS = new Dictionary<string, Func<string, string>>
{
    ["CooperateBot_toy"] = CooperateBot_toy,
    ["DefectBot_toy"] = DefectBot_toy,
    ["FairBot_toy"] = FairBot_toy,
    ["CUPOD_toy"] = CUPOD_toy,
    ["PrudentBot_toy"] = PrudentBot_toy,
};

// Representations textuelles DECLAREES : elles constituent l'argument lu par les delegates.
// Chaque source est un fragment C# VALIDE : les guillemets internes y sont echappes.
var SOURCES = new Dictionary<string, string>
{
    ["CooperateBot_toy"] = "string CooperateBot_toy(string otherSource) { return \"C\"; }",
    ["DefectBot_toy"] = "string DefectBot_toy(string otherSource) { return \"D\"; }",
    ["FairBot_toy"] = "string FairBot_toy(string otherSource) { if (otherSource.Contains(\"" + markerEscaped + "\")) return \"C\"; return \"D\"; }",
    ["CUPOD_toy"] = "string CUPOD_toy(string otherSource) { if (otherSource.Contains(\"" + markerEscaped + "\")) return \"C\"; return \"D\"; }",
    ["PrudentBot_toy"] = "string PrudentBot_toy(string otherSource) { if (otherSource.Contains(\"DefectBot_toy\") && !otherSource.Contains(\"CUPOD_toy\") && !otherSource.Contains(\"FairBot_toy\")) return \"D\"; if (otherSource.Contains(\"" + markerEscaped + "\")) return \"C\"; return \"D\"; }",
};

Console.WriteLine($"Bots declares : {BOTS.Count} ; sources declarees : {SOURCES.Count}");
foreach (var entry in SOURCES)
    Console.WriteLine($"{entry.Key} => {entry.Value}");

// Controle SUPERFICIEL des sources declarees : formes elementaires seulement
// (guillemets pairs, presence d'une clause de retour, prefixe coherent avec le nom du bot).
// Ce controle ne compile pas les fragments : il ne prouve donc pas leur validite syntaxique C#.
foreach (var entry in SOURCES)
{
    int quotes = entry.Value.Count(ch => ch == '"');
    if (quotes % 2 != 0)
        throw new InvalidOperationException($"Source declaree mal formee (guillemets impairs) : {entry.Key}");
    if (!entry.Value.Contains("return"))
        throw new InvalidOperationException($"Source declaree sans clause de retour : {entry.Key}");
    if (!entry.Value.StartsWith("string " + entry.Key))
        throw new InvalidOperationException($"Source declaree mal nommee : {entry.Key}");
}
Console.WriteLine("Controle superficiel des sources : guillemets pairs, clause de retour presente, nom coherent.");


Bots declares : 5 ; sources declarees : 5


CooperateBot_toy => string CooperateBot_toy(string otherSource) { return "C"; }


DefectBot_toy => string DefectBot_toy(string otherSource) { return "D"; }


FairBot_toy => string FairBot_toy(string otherSource) { if (otherSource.Contains("return \"C\"")) return "C"; return "D"; }


CUPOD_toy => string CUPOD_toy(string otherSource) { if (otherSource.Contains("return \"C\"")) return "C"; return "D"; }


PrudentBot_toy => string PrudentBot_toy(string otherSource) { if (otherSource.Contains("DefectBot_toy") && !otherSource.Contains("CUPOD_toy") && !otherSource.Contains("FairBot_toy")) return "D"; if (otherSource.Contains("return \"C\"")) return "C"; return "D"; }


Controle superficiel des sources : guillemets pairs, clause de retour presente, nom coherent.


## 4. Matrice de confrontation : trois phénomènes


In [4]:
// Illustration 4 — matrice 5 x 5 : vingt-cinq confrontations recalculees par le moteur
var rows = new List<(string A, string B, string Act, int PayoffA, int PayoffB, string Status)>();

foreach (string nameA in BOTS.Keys)
{
    foreach (string nameB in BOTS.Keys)
    {
        try
        {
            var result = engine.SimulatePayoff(BOTS[nameA], BOTS[nameB], SOURCES[nameA], SOURCES[nameB]);
            rows.Add((nameA, nameB, $"{result.ActionA}/{result.ActionB}", result.PayoffA, result.PayoffB, result.Status));
        }
        catch (SimulationTimeoutException)
        {
            rows.Add((nameA, nameB, "D/D", PayoffSelf("D", "D"), PayoffSelf("D", "D"), "timeout"));
        }
    }
}

Console.WriteLine($"{"A",-20} {"B",-20} {"act",-4} {"payoff_A",8} {"payoff_B",8} {"status",-8}");
foreach (var row in rows)
    Console.WriteLine($"{row.A,-20} {row.B,-20} {row.Act,-4} {row.PayoffA,8} {row.PayoffB,8} {row.Status,-8}");

if (rows.Count != 25)
    throw new InvalidOperationException($"La matrice doit contenir 25 confrontations, obtenu {rows.Count}.");
if (rows.Any(r => r.Status != "play"))
    throw new InvalidOperationException("Toutes les confrontations doivent rendre un verdict 'play' (verdict CALCULE) avec StepBudget=1000 et bots deterministes purs.");
if (engine.BotCallCount != 2 * rows.Count)
    throw new InvalidOperationException($"Les delegates doivent etre appeles une seule fois par confrontation : attendu {2 * rows.Count}, observe {engine.BotCallCount}.");
Console.WriteLine($"{rows.Count}/25 confrontations rendent un verdict 'play' (CALCULE dans la borne).");
Console.WriteLine($"Delegues appeles : {engine.BotCallCount} pour {rows.Count} confrontations (2 par confrontation, chacun une seule fois).");


A                    B                    act  payoff_A payoff_B status  


CooperateBot_toy     CooperateBot_toy     C/C         3        3 play    


CooperateBot_toy     DefectBot_toy        C/D         0        5 play    


CooperateBot_toy     FairBot_toy          C/C         3        3 play    


CooperateBot_toy     CUPOD_toy            C/C         3        3 play    


CooperateBot_toy     PrudentBot_toy       C/C         3        3 play    


... (18 outputs left out)

PrudentBot_toy       CUPOD_toy            C/C         3        3 play    


PrudentBot_toy       PrudentBot_toy       C/C         3        3 play    


25/25 confrontations rendent un verdict 'play' (CALCULE dans la borne).


Delegues appeles : 50 pour 25 confrontations (2 par confrontation, chacun une seule fois).


### Lecture de la matrice

Trois phénomènes apparaissent dans la matrice 5x5 ci-dessus. **Deux modèles coexistent et
il faut les distinguer** : (i) le verdict classique de la forme normale (section 1, « Le Dilemme du prisonnier classique ») — `(D, D)`
est l'unique équilibre de Nash en stratégies pures ; (ii) le verdict **calculé** par la
transparence des programmes — `MaxDepth = 3` borne les niveaux de descente du moteur et
`StepBudget = 1000` le nombre d'appels autorisés, puis le moteur évalue chaque confrontation.
Les phénomènes ci-dessous relèvent du second modèle : ils ne sont **pas** des équilibres de Nash du premier.

1. **Coopération mutuelle** : `FairBot_toy` face à `CUPOD_toy` ou `CooperateBot_toy` produit
   `(C, C)` avec payoff `(3, 3)`. La transparence du programme permet à chaque bot de détecter
   la clause `return "C"` de l'autre et de choisir `C` en conséquence.
2. **Exploitation** : `DefectBot_toy` face à `CooperateBot_toy` produit `(D, C)` avec payoff
   `(5, 0)`. Le bot patient (`CooperateBot_toy`) ne lit rien et se fait exploiter — l'asymétrie
   informationnelle met fin à la coopération, même dans un jeu one-shot.
3. **Défection paradoxale** : `PrudentBot_toy` face à `CUPOD_toy` détecte `CUPOD_toy` **et** la
   clause `return "C"` ; il joue donc `C`. Face à `DefectBot_toy` pur, il détecte `DefectBot_toy`
   sans `CUPOD_toy` ni `FairBot_toy` et joue `D`. La transparence révèle le type de l'adversaire :
   le bot patient n'est récompensé que si son intention coopérative est lisible dans le source.

Dans ce modèle, `(D, D)` n'est plus l'unique issue : la transparence du programme est un
mécanisme de signal, comme la réputation ou la menace de représailles (cf. Axelrod & Hamilton
1981 pour la thèse générale, voir header §0).

Les bornes `MaxDepth` et `StepBudget` limitent les niveaux de descente et les appels du moteur.
Depuis une descente partie de `depth = 0`, la condition `depth + 1 < MaxDepth` arrête la
récursion à `MaxDepth - 1` : la branche `depth` et l'exception de budget ne sont donc atteintes
que par injection d'une valeur limite à l'entrée.
Elles ne limitent pas la lecture textuelle effectuée par un delegate : chaque bot reçoit le
texte source complet de l'adversaire. Cette distinction décrit le comportement implémenté ;
le carnet n'attribue aucune garantie de sécurité à la lecture de ce texte.


## 5. Vérificateurs natifs : reproduire la matrice

Deux contrôles natifs distincts sont conservés dans le carnet :

- un **contrôle de cohérence interne**, qui recalcule les confrontations à partir des mêmes
  bots et compare le résultat à la matrice produite par le moteur ;
- une **table-oracle déclarative** de 25 paires, encodée à la main à partir de la
  spécification des bots, puis comparée au moteur.

Le second contrôle est structurellement distinct du premier : il ne rappelle ni les bots ni la
fonction de paiement du moteur. Il reste toutefois interne au livrable et ne constitue pas un
vérificateur externe indépendant.


Les deux contrôles qui suivent sont **internes au livrable** : aucun n'est un vérificateur
externe, aucun n'exécute un programme hors de ce carnet.

- Le **contrôle de cohérence interne** (cellule suivante) recalcule chaque paire avec les
  mêmes délégués et la même fonction `PayoffSelf` : il vérifie que le moteur et ce recalcul
  produisent la même matrice. Les cinq bots étant déterministes et sans état, c'est un
  contrôle de cohérence, pas une preuve d'indépendance d'implémentation.
- La **table-oracle déclarative** (cellule d'après) contient 25 paires écrites à la main
  depuis la spécification des bots ; elle ne rappelle ni les délégués ni `PayoffSelf`, et
  constitue donc le seul organe réellement distinct du livrable.

La source Python confie ce rôle à un **troisième organe** : un script de vérification
interne (`verify_program_games_table.py`) lancé dans un autre processus. Ce jumeau C# ne
l'appelle pas et ne le reproduit pas — aucun lancement de Python n'a lieu ici. Les deux
contrôles natifs ci-dessus sont les contrôles disponibles dans ce carnet, et leur accord
est ce qui fait de la matrice une mesure.


In [5]:
// Illustration 5 — verificateur de coherence INTERNE au moteur
// Ce verificateur reutilise BOTS, SOURCES et PayoffSelf du moteur principal.
// Il confirme la coherence interne, PAS comme organe independant du livrable.
// L'organe distinct est la table declarative de la cellule suivante.

(string ActA, string ActB, int PayA) IndependentPair(string nameA, string nameB)
{
    string actA = BOTS[nameA](SOURCES[nameB]);
    string actB = BOTS[nameB](SOURCES[nameA]);
    return (actA, actB, PayoffSelf(actA, actB));
}

int ok = 0;
foreach (var r in rows)
{
    var (actA, actB, pay) = IndependentPair(r.A, r.B);
    bool same = $"{actA}/{actB}" == r.Act && pay == r.PayoffA;
    if (!same)
        throw new InvalidOperationException($"Mismatch sur {r.A} vs {r.B}: moteur={r.Act}, verificateur_interne={actA}/{actB}");
    ok++;
}
if (ok != rows.Count)
    throw new InvalidOperationException($"Coherence interne incomplete : {ok}/{rows.Count}");
Console.WriteLine($"Coherence interne moteur : {ok}/{rows.Count} confrontations agree (trivialement, bots deterministes).");

Coherence interne moteur : 25/25 confrontations agree (trivialement, bots deterministes).


In [6]:
// Illustration 6 — second organe declaratif : table de verite litterale de 25 paires
// Table encodee a la main depuis la specification des cinq bots.
// Elle ne rappelle ni les delegates ni PayoffSelf : elle fournit un second controle natif.

var EXPECTED_TABLE_V2 = new Dictionary<string, (string ActA, string ActB, int PayA, int PayB)>
{
    ["CooperateBot_toy|CooperateBot_toy"] = ("C", "C", 3, 3),
    ["CooperateBot_toy|DefectBot_toy"] = ("C", "D", 0, 5),
    ["CooperateBot_toy|FairBot_toy"] = ("C", "C", 3, 3),
    ["CooperateBot_toy|CUPOD_toy"] = ("C", "C", 3, 3),
    ["CooperateBot_toy|PrudentBot_toy"] = ("C", "C", 3, 3),
    ["DefectBot_toy|CooperateBot_toy"] = ("D", "C", 5, 0),
    ["DefectBot_toy|DefectBot_toy"] = ("D", "D", 1, 1),
    ["DefectBot_toy|FairBot_toy"] = ("D", "D", 1, 1),
    ["DefectBot_toy|CUPOD_toy"] = ("D", "D", 1, 1),
    ["DefectBot_toy|PrudentBot_toy"] = ("D", "D", 1, 1),
    ["FairBot_toy|CooperateBot_toy"] = ("C", "C", 3, 3),
    ["FairBot_toy|DefectBot_toy"] = ("D", "D", 1, 1),
    ["FairBot_toy|FairBot_toy"] = ("C", "C", 3, 3),
    ["FairBot_toy|CUPOD_toy"] = ("C", "C", 3, 3),
    ["FairBot_toy|PrudentBot_toy"] = ("C", "C", 3, 3),
    ["CUPOD_toy|CooperateBot_toy"] = ("C", "C", 3, 3),
    ["CUPOD_toy|DefectBot_toy"] = ("D", "D", 1, 1),
    ["CUPOD_toy|FairBot_toy"] = ("C", "C", 3, 3),
    ["CUPOD_toy|CUPOD_toy"] = ("C", "C", 3, 3),
    ["CUPOD_toy|PrudentBot_toy"] = ("C", "C", 3, 3),
    ["PrudentBot_toy|CooperateBot_toy"] = ("C", "C", 3, 3),
    ["PrudentBot_toy|DefectBot_toy"] = ("D", "D", 1, 1),
    ["PrudentBot_toy|FairBot_toy"] = ("C", "C", 3, 3),
    ["PrudentBot_toy|CUPOD_toy"] = ("C", "C", 3, 3),
    ["PrudentBot_toy|PrudentBot_toy"] = ("C", "C", 3, 3),
};

int okV2 = 0;
foreach (var r in rows)
{
    string key = r.A + "|" + r.B;
    if (!EXPECTED_TABLE_V2.TryGetValue(key, out var expected))
        throw new KeyNotFoundException($"cle absente de l'oracle : {key}");
    bool same = expected.ActA + "/" + expected.ActB == r.Act
        && expected.PayA == r.PayoffA
        && expected.PayB == r.PayoffB;
    if (!same)
        throw new InvalidOperationException($"Mismatch independent_v2 sur {r.A} vs {r.B}: moteur={r.Act}, oracle=({expected.ActA},{expected.ActB},{expected.PayA},{expected.PayB})");
    okV2++;
}
if (okV2 != 25)
    throw new InvalidOperationException($"independent_v2: attendu 25/25, obtenu {okV2}/25");
Console.WriteLine($"Matrice reproductible (oracle declaratif v2) : {okV2}/25 confrontations agree.");
Console.WriteLine($"Table oracle : {EXPECTED_TABLE_V2.Count} paires, encodees a la main depuis la spec (independance structurelle).");
Console.WriteLine("Difference structurelle honnete : le verificateur Python externe de la source n'est ni appele ni reproduit en C# ; les deux controles natifs ci-dessus sont les controles disponibles dans ce jumeau.");

Matrice reproductible (oracle declaratif v2) : 25/25 confrontations agree.


Table oracle : 25 paires, encodees a la main depuis la spec (independance structurelle).


Difference structurelle honnete : le verificateur Python externe de la source n'est ni appele ni reproduit en C# ; les deux controles natifs ci-dessus sont les controles disponibles dans ce jumeau.


## 6. Trois états : preuve, absence dans la borne, non-termination


## 6. Trois états : preuve, absence dans la borne, non-termination


On distingue **trois états d'arrêt** :

- **`play`** : la confrontation produit une action et un payoff **calculés** dans la borne
  (`step_cap > 0` et `depth < MAX_DEPTH`). Le verbe « prouver » est réservé aux démonstrations
  formelles ; ici, le simulateur calcule le résultat d'une confrontation one-shot.
- **`depth`** : la profondeur transmise à l'entrée est `>= MaxDepth` (ici `depth = MaxDepth`) :
  le moteur entre alors dans sa branche de repli **avant** tout appel de bot et renvoie l'action
  par défaut `(D, D)` et le payoff `(1, 1)`. Ce n'est pas une preuve que `(D, D)` est
  l'équilibre : c'est le résultat conventionnel de la borne de profondeur. Depuis une descente
  partie de `depth = 0`, cette branche n'est **pas** atteinte (la condition
  `depth + 1 < MaxDepth` arrête la récursion à `MaxDepth - 1`) : le statut `depth` s'observe en
  injectant la profondeur limite à l'appel.
- **`timeout`** : l'exception `SimulationTimeoutException` est levée quand le budget transmis à
  l'entrée est déjà épuisé (`stepCap <= 0`). Le budget est décrémenté avant l'appel au bot ; un
  budget déjà épuisé déclenche donc l'exception sans exécuter le bot. La descente récursive ne
  produit pas ce cas d'elle-même : le budget décroît d'une unité par niveau et la condition de
  descente (`remaining >= 1`) arrête la descente avant épuisement.

`depth` fournit une valeur de repli, tandis que `timeout` interrompt l'appel : une exception
n'est donc pas une défection observée. Avec un budget positif, un bot qui ne termine pas
peut encore bloquer le noyau ; le budget ne borne pas la durée de cet appel.

Aucun de ces états ne constitue une preuve formelle d'équilibre ni une preuve que `(D, D)`
est le seul équilibre.

En C#, `MAX_DEPTH` devient la propriété mutable `MaxDepth`, initialisée à 3, et `STEP_BUDGET`
la propriété mutable `StepBudget`, initialisée à 1000. Ces deux compteurs restent distincts :
`MaxDepth` borne la profondeur de descente du moteur, tandis que `StepBudget` borne le nombre
d'appels autorisés. Ni l'un ni l'autre ne borne la durée d'un delegate qui ne termine pas ni la
lecture du texte source complet qu'il reçoit.


In [7]:
// Illustration 7 — demonstration des trois etats via budget explicitement serre
// (1) depth : l'appel est effectue avec depth=MaxDepth -> le moteur entre dans sa branche de repli, status="depth"
var depthResult = engine.SimulatePayoff(
    CooperateBot_toy, CooperateBot_toy, SOURCES["CooperateBot_toy"], SOURCES["CooperateBot_toy"],
    engine.StepBudget, engine.MaxDepth);
Console.WriteLine($"depth=MaxDepth -> act={depthResult.ActionA}/{depthResult.ActionB}, payoff=({depthResult.PayoffA}, {depthResult.PayoffB}), status={depthResult.Status}");
if (depthResult.Status != "depth")
    throw new InvalidOperationException($"attendu depth, obtenu {depthResult.Status}");
if (depthResult.ActionA != "D" || depthResult.ActionB != "D")
    throw new InvalidOperationException("verdict par defaut doit etre D/D");
if (depthResult.PayoffA != 1 || depthResult.PayoffB != 1)
    throw new InvalidOperationException("payoff du verdict par defaut doit etre (1, 1)");

// (2) timeout : on force step_cap=0 -> SimulationTimeoutException LEVEE
bool timeoutLeve = false;
try
{
    engine.SimulatePayoff(CooperateBot_toy, CooperateBot_toy, SOURCES["CooperateBot_toy"], SOURCES["CooperateBot_toy"], 0, 0);
}
catch (SimulationTimeoutException e)
{
    timeoutLeve = true;
    Console.WriteLine($"step_cap=0 -> SimulationTimeout levee : {e.Message}");
}
if (!timeoutLeve)
    throw new InvalidOperationException("SimulationTimeout doit etre levee sur step_cap=0");

// (3) play : verdict CALCULE (pas "preuve") dans la borne
var playResult = engine.SimulatePayoff(
    CooperateBot_toy, CooperateBot_toy, SOURCES["CooperateBot_toy"], SOURCES["CooperateBot_toy"]);
Console.WriteLine($"verdict normal -> act={playResult.ActionA}/{playResult.ActionB}, payoff=({playResult.PayoffA}, {playResult.PayoffB}), status={playResult.Status}");
if (playResult.Status != "play")
    throw new InvalidOperationException($"attendu play (verdict CALCULE), obtenu {playResult.Status}");
if (playResult.ActionA != "C" || playResult.ActionB != "C")
    throw new InvalidOperationException("CooperateBot_toy vs CooperateBot_toy -> C/C");
if (playResult.PayoffA != 3 || playResult.PayoffB != 3)
    throw new InvalidOperationException("payoff C/C doit etre (3, 3)");

// (4) consommation REELLE du budget : stepCap=1 -> aucune descente recursive autorise
var tightEngine = new ProgramGameEngine(PayoffSelf) { MaxDepth = 3, StepBudget = 1 };
var tight = tightEngine.SimulatePayoff(
    CooperateBot_toy, CooperateBot_toy, SOURCES["CooperateBot_toy"], SOURCES["CooperateBot_toy"], 1, 0);
Console.WriteLine($"stepCap=1 -> act={tight.ActionA}/{tight.ActionB}, payoff=({tight.PayoffA}, {tight.PayoffB}), status={tight.Status}, descentes={tightEngine.DescentCount}, budgets observes=[{string.Join(", ", tightEngine.StepCapsObserved)}]");
if (tightEngine.DescentCount != 0)
    throw new InvalidOperationException($"stepCap=1 ne doit autoriser aucune descente recursive, observe {tightEngine.DescentCount}");
if (tightEngine.StepCapsObserved.Count != 1 || tightEngine.StepCapsObserved[0] != 1)
    throw new InvalidOperationException("le budget transmis doit etre le budget initial decremente, jamais stepCap+1");
if (tightEngine.DepthsObserved.Count != 1 || tightEngine.DepthsObserved[0] != 0)
    throw new InvalidOperationException("la profondeur transmise doit etre 0 au premier appel");
if (tight.Status != "play")
    throw new InvalidOperationException($"stepCap=1 doit produire un verdict play, obtenu {tight.Status}");
if (tight.ActionA != "C" || tight.ActionB != "C")
    throw new InvalidOperationException($"stepCap=1 doit produire C/C, obtenu {tight.ActionA}/{tight.ActionB}");
if (tight.PayoffA != 3 || tight.PayoffB != 3)
    throw new InvalidOperationException($"stepCap=1 doit produire le payoff (3, 3), obtenu ({tight.PayoffA}, {tight.PayoffB})");

Console.WriteLine("Trois etats distincts : 'play' (verdict CALCULE), 'depth' (verdict par defaut borne), 'timeout' (exception).");
Console.WriteLine("Aucun n'est une preuve formelle d'equilibre -- tous sont des verdicts d'arret dans la borne.");

depth=MaxDepth -> act=D/D, payoff=(1, 1), status=depth


step_cap=0 -> SimulationTimeout levee : step_cap epuise avant verdict (step_cap_initial=1000)


verdict normal -> act=C/C, payoff=(3, 3), status=play


stepCap=1 -> act=C/C, payoff=(3, 3), status=play, descentes=0, budgets observes=[1]


Trois etats distincts : 'play' (verdict CALCULE), 'depth' (verdict par defaut borne), 'timeout' (exception).


Aucun n'est une preuve formelle d'equilibre -- tous sont des verdicts d'arret dans la borne.


## 7. Comparaison : répétition/Axelrod, preuve bornée, simulation


| Mécanisme | Issue du PD one-shot | Source |
|---|---|---|
| **Forme normale classique** | `(D, D)` unique Nash par dominance stricte | Osborne §3.2 |
| **Répétition (Axelrod 1980)** | `(C, C)` si ombrage du futur (δ ≈ 1) | `GameTheory-06` Evolution of Trust ; Axelrod & Hamilton 1981 |
| **Preuve formelle (Lean)** | `(C, C)` par démonstration dans le calcul des constructions | `GameTheory-06b` Lean |
| **Transparence des programmes** (ce notebook) | `(C, C)` par lecture de la source, dans la borne `step_cap` | Ce notebook -- `GameTheory-06e` |
| **Simulation multi-graines** | distribution empirique | `GameTheory-06c` Folk theorem |

Quatre mécanismes convergent vers `(C, C)` mais par des chemins distincts : la répétition
donne du poids au futur, la preuve formelle établit l'équilibre dans le calcul, la transparence
donne accès à l'intention, la simulation borne l'incertitude. Voir header §0 pour les
références complètes (Shoham-Leyton-Brown §3.4.2 pour la paramétrisation canonique du PD,
Osborne §3.2 pour la dominance stricte, Axelrod & Hamilton 1981 pour la dynamique évolutive,
Critch-Dennis-Russell 2022 §3 pour le cadre d'inspection mutuelle).

La ligne « transparence des programmes » décrit ici un **verdict calculé** dans la borne du
moteur, non une propriété établie : la borne `step_cap` compte les appels du moteur, elle ne
borne pas la lecture du texte source que reçoit un delegate.


## 8. Limites et Open Problems


## 8. Limites et Open Problems


- **Folk theorems des jeux-programmes — trois régimes, une frontière** (Cooper-Oesterheld-Conitzer 2024, arXiv:2412.14570). La comparaison de la section 7 (répétition / preuve / simulation) se prolonge en littérature : (1) **Tennenholtz (2004)** prouve un folk theorem pour les jeux-programmes — avec des programmes capables de lire leur adversaire, tout profil de paiements faisable et individuellement rationnel devient atteignable en équilibre (Theorem 1, p. 3 de Cooper et al.) ; (2) pour les programmes **simulationistes** (`εGroundedπBot`), Cooper et al. retrouvent un folk theorem complet **avec aléa partagé** (Theorem 5, p. 5 ; Corollary 1, p. 6) ; (3) **sans aléa partagé, le folk theorem complet échoue** (Propositions 1-2, p. 7) — seules les utilités additivement séparables le recouvrent partiellement (Theorem 7, p. 8), et il ne tient pas pour les programmes simulationistes non corrélés (p. 9). La frontière aléa partagé / sans aléa partagé est une limite **de fond** du mécanisme simulation, à distinguer des bornes opérationnelles ci-dessous.
- **Open Problem 3 de Critch-Dennis-Russell 2022** (cf. arXiv:2208.07006 p. 18) : la conjecture `DUPOC(k)` vs `CUPOD(k)` sur l'itération k de l'inspection mutuelle reste **explicitement ouverte** dans la littérature. Ce notebook l'illustre mais ne la résout pas — ne jamais la présenter comme un exercice à solution attendue. Les bots `_toy` du notebook ne sont **pas** des reproductions de `DUPOC` ou `CUPOD` (cf. header §0 : Critch-Dennis-Russell 2022 §3 pour la définition publiée).
- **Bornes opérationnelles** : `MaxDepth = 3` et `StepBudget = 1000` sont des choix opérationnels, et les deux propriétés sont mutables : les augmenter peut faire émerger des comportements différents (cf. `GameTheory-06c`, Folk theorem, pour la discussion). Si `step_cap <= 0` à l'entrée, `SimulationTimeoutException` est levée immédiatement. Sinon, le bot est appelé directement et un bot non terminant bloque le noyau .NET Interactive — il n'y a pas d'isolation par cellule dans un notebook. Depuis une descente partie de `depth = 0`, la condition `depth + 1 < MaxDepth` arrête la récursion à `MaxDepth - 1` et le budget décroît d'une unité par niveau : les états `depth` et `timeout` sont donc des cas limites obtenus en injectant la valeur limite à l'entrée, non des issues produites spontanément par la récursion.
- **Ce que les bornes ne couvrent pas** : `MaxDepth` borne la profondeur de descente du moteur et `StepBudget` borne le nombre d'appels autorisés. Ni l'un ni l'autre ne borne la **lecture textuelle** faite par un delegate : chaque bot reçoit le texte source **complet** de l'adversaire, sans troncature. Le carnet ne repose sur aucune propriété de sécurité attachée à cette lecture — il ne décrit que le comportement implémenté.
- **Inspection par chaîne textuelle** : `PrudentBot_toy` inspecte par `Contains` (`System.String.Contains`). Un adversaire qui obfusque sa source (ex. `chr(67)` au lieu de `"C"`) trompe l'inspection. Voir `GameTheory-06b` Lean pour la formalisation, et Critch-Dennis-Russell 2022 §3 pour la discussion publiée de ces mécanismes.


## À retenir

- **La lecture du programme change le jeu.** Sur la matrice mesurée au début du carnet (`T=5, R=3, P=1, S=0`), `D` domine `C` strictement et `(D, D)` est l'unique équilibre de Nash. La matrice finale contient pourtant **seize** confrontations en `(C, C)` — dont les quatre de la ligne `FairBot_toy`, contre `CooperateBot`, `FairBot`, `CUPOD` et `PrudentBot`, paiement mesuré `(3, 3)`. Ce sont les bots, pas la matrice, qui ont changé.
- **La coopération mesurée ne vient que de la transparence réciproque.** Face à `DefectBot_toy`, `FairBot_toy` retombe à `(D, D)` avec paiement `(1, 1)` : lire le programme adverse ne sert à rien si l'adversaire ne coopère conditionnellement à rien. La ligne du folk theorem de Tennenholtz (2004) cite exactement cette condition — des programmes capables de se lire mutuellement.
- **Deux contrôles natifs d'accord, pas un seul.** La matrice 25/25 verdicts `play` est produite par le moteur, puis recoupée par le contrôle de cohérence interne (`25/25 confrontations agree`) et par la table-oracle déclarative de 25 paires (`25/25 confrontations agree`). Un seul de ces contrôles aurait suffi à afficher des nombres ; leur accord est ce qui fait de la matrice une mesure. La source Python ajoute un troisième organe, un vérificateur externe exécuté dans un autre processus ; ce jumeau C# ne l'appelle pas et ne le reproduit pas — il conserve les deux contrôles natifs ci-dessus.
- **Trois états d'arrêt, aucun n'est une preuve.** `play`, `depth`, `timeout` disent seulement comment la confrontation s'est terminée sous `MaxDepth = 3` et `StepBudget = 1000` — « Aucun n'est une preuve formelle d'équilibre », comme le conclut le carnet. La preuve formelle vit dans le jumeau Lean (`Dynamics-02`, `Dynamics-04b`), pas dans la simulation.


## Vérifiez votre compréhension

1. **La borne qui fabrique une issue.** La section 6 imprime `depth=MaxDepth -> act=D/D, payoff=(1, 1), status=depth`. Que montre cette ligne, et quelle conclusion serait fausse ?

   <details><summary>Réponse</summary>

   **Réponse.** Elle montre que l'appel a été effectué **avec `depth = MaxDepth`** : le moteur entre alors dans sa branche de repli et renvoie l'action par défaut `D` avec le payoff `(1, 1)` et le statut `depth`. Ce n'est pas une confrontation qui a atteint la borne par récursion — la démonstration injecte directement la profondeur (une descente partie de `depth = 0` s'arrête à `MaxDepth - 1` et n'atteindrait jamais cette branche). En conclure que « les programmes concernés trahissent » serait faux — leur issue n'a pas été calculée du tout. C'est pourquoi le statut accompagne le paiement : `play` = verdict calculé, `depth` = repli sur la borne, `timeout` = exception levée. Les 25 confrontations de la matrice portent toutes `play` ; cette ligne-ci est un témoin négatif, pas une mesure. Ces bornes portent sur les **appels du moteur** ; elles ne limitent pas la **lecture textuelle** faite par un délégué, qui reçoit toujours le texte source complet de l'adversaire.

   </details>

2. **Deux contrôles natifs d'accord — sur quoi exactement ?** Le carnet affiche `Coherence interne moteur : 25/25 confrontations agree`, puis `Matrice reproductible (oracle declaratif v2) : 25/25`. Que contrôle cet accord, et que ne contrôle-t-il pas ?

   <details><summary>Réponse</summary>

   **Réponse.** Il contrôle l'**implémentation** : deux lectures distinctes — le moteur, et une table de 25 paires encodée à la main depuis la spécification des bots — rendent la même matrice. Il ne contrôle pas la **sémantique du modèle borné** : les deux peuvent s'accorder sur une matrice que le modèle calcule correctement et qui ne reproduit pourtant pas le théorème cité. L'accord dit « la matrice est bien celle que ce modèle calcule », jamais « ce modèle calcule ce que la source annonce » — la distinction est celle que la section 8 prolonge (Cooper-Oesterheld-Conitzer 2024). La source Python ajoute un vérificateur externe ; le jumeau C# s'en passe et ne prétend donc pas au même nombre d'organes.

   </details>

3. **`D` domine `C`, et la matrice porte seize `(C, C)`.** La section 1 imprime `D domine C (PD stricte). Equilibre de Nash unique : (D,D).`, et la matrice finale contient seize confrontations en `C/C` au paiement `(3, 3)`. Comment les deux tiennent-ils ensemble ?

   <details><summary>Réponse</summary>

   **Réponse.** Parce qu'ils ne portent pas sur le même objet. La dominance de `D` sur `C` est une propriété de la **matrice de paiement** (`T=5, R=3, P=1, S=0`) : elle compare deux actions à action adverse fixée, et elle reste vraie. La matrice mesurée compare des **programmes** : `FairBot_toy` ne choisit pas `C` dans l'abstrait, il le choisit après avoir lu `return "C"` dans le source de l'adversaire. La dominance ne dit rien d'un joueur dont l'action dépend du programme adverse — c'est le point du carnet : la transparence change les actions, pas les paiements. Contre `DefectBot_toy`, `FairBot_toy` retombe d'ailleurs à `D/D` : sans coopération conditionnelle en face, l'issue redevient celle que la dominance prédit.

   </details>


## Exercices

Quatre exercices répartis, à traiter dans l'ordre. Chacun porte un stub compilable : le code
s'exécute sans erreur, mais la partie à écrire est signalée par un commentaire `// TODO etudiant`.
Les exercices 1 et 3 s'appuient sur des éléments déjà présents dans le carnet (délégués, dictionnaire
`BOTS`, dictionnaire `SOURCES`, propriétés `MaxDepth` et `StepBudget` du moteur `engine`).


### Exercice 1 — Bot inédit : `TitForTatBot`

Implémentez `TitForTatBot` qui joue `C` au premier tour et recopie l'action de l'adversaire
au tour suivant. Vous aurez besoin de **stocker l'historique** entre deux appels : la signature
du délégué `Func<string, string>` ne suffit pas. Étendez `SimulatePayoff` (ou créez une
variante `SimulateRepeated`) qui passe l'historique comme argument supplémentaire.

Indice : la transparence ne s'applique plus au même jeu — `TitForTat` est une stratégie
des jeux répétés. Vous pouvez combiner les deux notebooks (`GameTheory-06` et `06e`) pour
tester votre variante.


In [8]:
// Exercice 1 -- stub a completer : TitForTatBot
Func<string, string> TitForTatBot_toy = otherSource => "";   // TODO etudiant : renvoyer "C" au premier tour, puis recopier l'action adverse du tour precedent

Console.WriteLine("Exercice 1 a completer.");

Exercice 1 a completer.


### Exercice 2 — Contre-exemple de robustesse

Trouvez un bot `X` tel que `FairBot` face à `X` **n'obtient pas** `(C, C)`. Justifiez en
montrant la chaîne textuelle que `FairBot` inspecte et pourquoi elle est silencieuse.

Indice : un bot qui retourne `C` conditionnellement à un **autre** signal que la présence
de `return "C"` peut déjouer l'inspection par chaîne.


In [9]:
// Exercice 2 -- stub a completer : bot X contre FairBot_toy
Func<string, string> X_contre_exemple_toy = otherSource => "";   // TODO etudiant : renvoyer une action qui ne depend pas seulement de la presence de 'return "C"'

Console.WriteLine("Exercice 2 a completer.");

Exercice 2 a completer.


### Exercice 3 — Variation de borne

`MaxDepth` est une **propriété mutable** du moteur `engine`, initialisée à 3. Choisissez une
valeur différente de 3, modifiez cette propriété, puis reconstruisez la matrice avec
`engine.SimulatePayoff`.

Produisez un rapport comparant les deux exécutions : nombre de confrontations, actions, gains,
statuts et traces de profondeur et de budget. Pour chaque différence observée, indiquez le niveau
auquel elle apparaît et justifiez-la à partir des valeurs effectivement transmises par le
moteur. Si une grandeur ne change pas, constatablez-le à partir de la comparaison ; ne déduisez
pas l'effet de la borne avant d'avoir exécuté les deux variantes.

Le stub doit permettre de choisir librement `depthMax`, de recalculer toute la matrice et de
produire ce rapport sans table de résultats préenregistrée.

In [10]:
// Exercice 3 -- stub a completer : variation de MAX_DEPTH
Func<int, List<(string A, string B, string Act, int PayoffA, int PayoffB, string Status)>> matrice_profondeur = depthMax => new List<(string A, string B, string Act, int PayoffA, int PayoffB, string Status)>();   // TODO etudiant : reconstruire la matrice avec un moteur dont MaxDepth vaut depthMax

Console.WriteLine("Exercice 3 a completer.");

Exercice 3 a completer.


### Exercice 4 — `payoff_self_v2` : heuristique non triviale

Le moteur `simulate_payoff`, défini dans la section 2 « ProgramAgent », utilise actuellement un `payoff_self` qui rend le
gain de l'agent en fonction des actions `(a, b)` des deux joueurs — matrice canonique
`(C,C)->(3,3)`, `(C,D)->(0,5)`, `(D,C)->(5,0)`, `(D,D)->(1,1)`. C'est l'unique source de
paiement du notebook. La transparence du programme ne modifie que les actions, pas la matrice
de paiement.

**Travail demandé** : implémenter `payoff_self_v2(my_action, other_action, other_source)`
qui prend en compte non seulement les actions `(a, b)`, mais **aussi** le source de l'adversaire.
Une heuristique non triviale peut, par exemple :

- ajouter une **pénalité** quand l'adversaire joue `D` après que sa source contient
  `return "C"` (incohérence : il trahit sa coopération annoncée dans le code) ;
- ajouter un **bonus** quand l'adversaire joue `C` et que sa source ne contient pas
  `return "C"` (choix délibéré, pas retour inconditionnel) ;
- moduler le gain selon la présence d'un commentaire `# cooperatif` ou `# defection` dans
  le source, indépendamment des actions observées.

**Contraintes** :

1. La signature de `payoff_self_v2` DOIT accepter `(my_action, other_action, other_source)` —
   le troisième argument est le **source** (`str`), pas l'objet bot. La fonction dépend ainsi
   de la représentation de l'adversaire.
2. Comparez la matrice obtenue avec la matrice canonique de la section 4. Expliquez quelles
   confrontations changent et si `(D, D)` apparaît encore. La règle de bonus ou de pénalité
   doit être déclarée et appliquée de façon cohérente ; il n'y a pas de résultat imposé.
3. Re-exécutez le notebook de bout en bout après avoir modifié le code, par exemple depuis
   son dossier avec `papermill Dynamics-04-Open-Source-Game-Theory-CSharp.ipynb sortie.ipynb`.

**Indice** : distinguer `payoff` de `payoff_self_v2` revient à étudier un jeu dont les gains
peuvent dépendre de la représentation du programme adverse, plutôt qu'uniquement des actions.
La théorie des jeux algorithmiques formalise ce type d'information sur les programmes ; voir
Shoham-Leyton-Brown, chapitre 7, pour le cas général.

En C#, le troisième argument reste une chaîne `string` : le délégué doit recevoir la
**représentation** de l'adversaire, pas une référence au delegate.


In [11]:
// Exercice 4 -- stub a completer : payoff_self_v2
Func<string, string, string, int> payoff_self_v2 = (myAction, otherAction, otherSource) => 0;   // TODO etudiant : heuristique prenant en compte le source de l'adversaire

Console.WriteLine("Exercice 4 a completer.");

Exercice 4 a completer.


## Pour aller plus loin

**Carnets voisins.**

- [Dynamics-04b-FairBot-Loeb-Lean](Dynamics-04b-FairBot-Loeb-Lean.ipynb) : le même FairBot côté Lean — là où la simulation borne la profondeur à 3, la preuve formelle ne la borne pas.
- [Dynamics-02-Jeux-Repetes-Lean-Python](Dynamics-02-Jeux-Repetes-Lean-Python.ipynb) : la formalisation de l'inspection mutuelle citée par la section 8 (chaîne textuelle vs. sémantique), côté jeux répétés en Lean.
- [Dynamics-05-Agents-Bornes-CSharp](Dynamics-05-Agents-Bornes-CSharp.ipynb) : les bornes opérationnelles (`MaxDepth`, budgets de pas) discutées pour elles-mêmes.
- [Dynamics-08-Institutions-Transparentes-CSharp](Dynamics-08-Institutions-Transparentes-CSharp.ipynb) : la transparence comme institution, au-delà des programmes individuels.

**Lectures.**

- Critch, B., Dennis, M. & Russell, S. (2022). « Cooperative and uncooperative institution designs: Surprises and problems in open-source game theory ». arXiv:2208.07006 — la définition publiée de `CUPOD`/`DUPOC` (§3), que les bots `_toy` de ce carnet n'imitent pas.
- Cooper, A., Oesterheld, C. & Conitzer, V. (2024). *Program equilibrium without shared randomness*. arXiv:2412.14570 — les folk theorems des jeux-programmes et la frontière aléa partagé / sans aléa partagé (section 8 du carnet).
- Tennenholtz, M. (2004). *Program equilibrium*. Games and Economic Behavior — le théorème fondateur que la matrice finale illustre.
- Shoham, Y. & Leyton-Brown, K. (2009). *Multiagent Systems*, §3.4.2 — le cadre des équilibres en stratégies dominantes, avant la lecture de programme.

**État de l'art ouvert (comparaison du 2026-11-16, issue #26985).**

- maisi.org — *Open-Source Game Theory* (page de direction de recherche) : la taxonomie DUPOD/CUPOD revisitée, le Löb borné paramétrique (f(k) ≳ log k) et l'Open Problem 3 (l'issue de `DUPOC(k) × CUPOD(k)` pour `k` grand, ouverte — l'exercice 4 du carnet 06 l'explore sous la borne moteur).
- transhumanist_atom_understander — *Payorian cooperation is easy with Kripke frames*, Alignment Forum, 2026-03-09 (https://www.alignmentforum.org/posts/LaCP6WyNzX8kiZn3w/) : la coopération sans Löb via le lemme de Payor, prouvée sémantiquement sur des cadres de Kripke finis (2 mondes de profondeur contre 4 pour la route löbienne).
- Duclaux, Formenti, Cobben, Zhang, Schölkopf & Jin (2026) — *open-source-game-theory* : mécanisation Lean complète de l'OSGT (PA-S' sur lib Foundation, Löb borné inconditionnel, τ-bots, EGT des populations). Repo `ColombanD/open-source-game-theory`, pin `main@c7f0ed76773c` — **sans licence** : citation seule, aucune réutilisation de code.
- Math-for-AI-Safety — *MAIS, agendas* A1–A8 (A1 : Löb borné quantitatif — coût minimal F(k,n) et seuils de coopération pour agents à ressources bornées ; MAIS-O16 : budget de coopération minimal de FairBot). Repo `Math-for-AI-Safety/MAIS`, pin `1cb9f003d111`.
- Robertboy18 — *TorchLean-Verified-Examples*, `week-07-program-equilibrium` et `week-08-bounded-lob` : les agents modaux de Barasz et al. (2014) évalués sur la chaîne des mondes GL, et le Löb borné paramétrique prouvé. Repo `Robertboy18/TorchLean-Verified-Examples`, pin `5afa0589a4d9`.

> Carnet issu de l'EPIC #26534.
